# Sistemas Expertos (


Este notebook complementa lo visto en:

- Unidad I: fundamentos de IA y agentes inteligentes
- Unidad II: entrada/salida por voz en sistemas simbólicos
- Unidad III: representación del conocimiento y razonamiento (lógica, encadenamiento, ontologías)

Aquí nos enfocamos en **Sistemas Expertos**:

- Definición y componentes (base de conocimiento, motor de inferencia, interfaz).
- Tipos de reglas (producción, heurísticas, condicionales).
- Ciclo de inferencia (coincidencia → selección → ejecución).
- Construcción de un sistema experto simple en Python.

> Caso práctico sugerido (coherente con lo anterior):  
> **“Detector de plantas + diagnóstico de enfermedades + cuidados”** con reglas y explicabilidad.

---

## 0) Objetivos de aprendizaje de la Unidad IV

Al terminar, deberías poder:

1) **Explicar** qué es un sistema experto y cuáles son sus componentes.  
2) **Distinguir** tipos de reglas: producción, heurísticas, condicionales.  
3) **Describir** el ciclo de inferencia (match/select/execute) y por qué existe.  
4) **Implementar** un mini sistema experto en Python que:
   - reciba hechos observados,
   - aplique reglas,
   - obtenga conclusiones,
   - y entregue explicación (rastro de reglas activadas).

---

## 1) ¿Qué es un Sistema Experto?

Un **Sistema Experto (SE)** es un sistema que resuelve problemas “como un experto” usando:

- **Conocimiento explícito** (hechos y reglas)
- **Razonamiento** (métodos de inferencia)

### 1.1 Componentes clásicos

1) **Base de conocimiento (BC)**  
   - reglas (si-entonces), ontologías, casos, heurísticas.

2) **Memoria de trabajo (MT)**  
   - hechos del caso actual (lo observado o ingresado).

3) **Motor de inferencia (MI)**  
   - mecanismo que aplica reglas a hechos para inferir nuevos hechos.

4) **Interfaz**  
   - cómo interactúa el usuario con el sistema (texto, web, voz, etc.)

5) (Opcional) **Módulo de explicación**  
   - responde: “¿por qué concluiste eso?” y “¿cómo llegaste a esa conclusión?”

---

## 2) Tipos de reglas

### 2.1 Reglas de producción
Forma típica:

> **SI** condiciones **ENTONCES** conclusión/acción

Ejemplo:
- SI `moho_blanco` Y `hojas_deformadas` ENTONCES `oidio`.

### 2.2 Reglas heurísticas
Una heurística es una “regla práctica” (no necesariamente perfecta), por ejemplo:
- “Si hay exceso de humedad, prioriza sospecha de hongos”.

Suelen ser:
- aproximadas,
- basadas en experiencia,
- útiles cuando el dominio es incierto.

### 2.3 Reglas condicionales / de control
Reglas que controlan el flujo o prioridad:
- “Si dos reglas compiten, elige la más específica”.
- “Si ya se diagnosticó X, no ejecutes diagnósticos incompatibles”.

Esto se relaciona con la **estrategia de resolución de conflictos**.

---

## 3) Ciclo de inferencia (Match → Select → Execute)

Un motor de inferencia, por lo general, corre un ciclo:

1) **Coincidencia (Match)**  
   - busca qué reglas pueden activarse con los hechos actuales.

2) **Selección (Select)**  
   - si hay varias reglas aplicables, decide cuál ejecutar primero.  
   - aquí entran prioridades, especificidad, recencia, etc.

3) **Ejecución (Execute / Fire)**  
   - aplica la regla: agrega hechos nuevos (o ejecuta acciones).

Esto se repite hasta:
- no haber reglas aplicables (saturación), o
- alcanzar una meta (en backward chaining).

---

## 4) Construcción de un Sistema Experto simple en Python

Vamos a implementar un SE con:
- Hechos (Fact)
- Reglas (Rule)
- Motor de inferencia (Forward chaining)
- Módulo de explicación (rastro de reglas activadas)

### Caso: Plantas (rasgos + síntomas + contexto) → (tipo, enfermedad, cuidados)

In [1]:
from dataclasses import dataclass
from typing import List, Tuple, Set, Dict, Optional, Callable, Any
import itertools

In [2]:
# -----------------------------
# 4.1 Estructuras base
# -----------------------------

@dataclass(frozen=True)
class Fact:
    sujeto: str
    predicado: str
    objeto: str

@dataclass
class Rule:
    nombre: str
    antecedentes: List[Tuple[str, str, str]]   # (sujeto(var o constante), predicado, objeto)
    consecuentes: List[Tuple[str, str, str]]   # idem
    prioridad: int = 0                         # para resolución de conflictos (select)
    condicion_extra: Optional[Callable[[Dict[str,str], Set[Fact]], bool]] = None

def fact(s, p, o) -> Fact:
    return Fact(str(s), str(p), str(o))

### 4.2 Coincidencia (Match): unificación básica

- Permitimos variables de sujeto como `"$x"`.
- Si el antecedente es `("$x", "tiene_sintoma", "moho_blanco")`,
  puede coincidir con el hecho `("p1", "tiene_sintoma", "moho_blanco")`
  ligando `"$x" = "p1"`.

In [3]:
def match_pattern(pattern: Tuple[str,str,str], f: Fact, bindings: Dict[str,str]) -> Optional[Dict[str,str]]:
    ps, pp, po = pattern
    if pp != f.predicado or po != f.objeto:
        return None

    # sujeto variable
    if ps.startswith("$"):
        var = ps
        if var in bindings and bindings[var] != f.sujeto:
            return None
        nb = dict(bindings)
        nb[var] = f.sujeto
        return nb

    # sujeto constante
    if ps == f.sujeto:
        return dict(bindings)

    return None

def find_bindings_for_antecedents(antecedents: List[Tuple[str,str,str]], facts: Set[Fact]) -> List[Dict[str,str]]:
    results = []

    def backtrack(i: int, bindings: Dict[str,str]):
        if i == len(antecedents):
            results.append(bindings)
            return
        pat = antecedents[i]
        for f in facts:
            nb = match_pattern(pat, f, bindings)
            if nb is not None:
                backtrack(i+1, nb)

    backtrack(0, {})
    return results

### 4.3 Ejecución (Execute): producir nuevos hechos

Una regla dispara si:
- existe al menos una asignación de variables (bindings) que satisface todos sus antecedentes.
- (opcional) se cumple `condicion_extra`.

Al disparar:
- se generan consecuentes como nuevos hechos.

In [4]:
def apply_rule(rule: Rule, facts: Set[Fact]) -> List[Tuple[Fact, Dict[str,str]]]:
    fired = []
    bindings_list = find_bindings_for_antecedents(rule.antecedentes, facts)

    for b in bindings_list:
        if rule.condicion_extra and not rule.condicion_extra(b, facts):
            continue

        for (cs, cp, co) in rule.consecuentes:
            sujeto = b.get(cs, cs) if cs.startswith("$") else cs
            nf = fact(sujeto, cp, co)
            if nf not in facts:
                fired.append((nf, b))
    return fired

### 4.4 Selección (Select): resolución de conflictos (prioridades)

Si múltiples reglas pueden disparar, elegimos:
- mayor **prioridad** primero
- y si empatan, orden por nombre (estable, didáctico)

En sistemas reales, pueden usarse:
- especificidad (más condiciones gana),
- recencia (hechos más nuevos),
- agenda (encadenamiento controlado).

In [5]:
def forward_chaining(
    rules: List[Rule],
    facts: Set[Fact],
    max_iter: int = 50,
    verbose: bool = True
) -> Tuple[Set[Fact], List[Dict[str, Any]]]:
    all_facts = set(facts)
    trace: List[Dict[str, Any]] = []

    for it in range(1, max_iter+1):
        # MATCH: para cada regla, ver qué nuevos hechos produciría
        agenda = []
        for r in rules:
            fired = apply_rule(r, all_facts)
            for (nf, b) in fired:
                agenda.append((r.prioridad, r.nombre, r, nf, b))

        if not agenda:
            if verbose:
                print(f"Detenido: no hay reglas aplicables (iter {it}).")
            break

        # SELECT: ordenamos por prioridad desc, luego nombre
        agenda.sort(key=lambda x: (-x[0], x[1]))

        # EXECUTE: en este SE didáctico, ejecutamos UNA por iteración (para ver el ciclo)
        prio, rname, r, nf, b = agenda[0]
        all_facts.add(nf)

        step = {
            "iter": it,
            "regla": r.nombre,
            "prioridad": r.prioridad,
            "bindings": b,
            "nuevo_hecho": nf
        }
        trace.append(step)

        if verbose:
            print(f"[Iter {it}] FIRE: {r.nombre} (prio={r.prioridad}) -> {nf} con {b}")

    return all_facts, trace

---

## 5) Base de Conocimiento: reglas del dominio (plantas)

Construimos una base de conocimiento pequeña y ampliable.

### 5.1 Reglas de identificación (detector por rasgos)
- Si `tiene_espinas` y `tallo_suculento` → `cactus`
- Si `hojas_gruesas` y `roseta` → `suculenta`
- Si `tallo_lenoso` y `hojas_ovales` → `ficus`

### 5.2 Reglas de diagnóstico (síntomas → enfermedad)
- `moho_blanco ∧ hojas_deformadas` → `oidio`
- `manchas_circulares ∧ puntos_negros` → `mancha_foliar`
- `hojas_caidas ∧ amarilleo ∧ exceso_humedad` → `pudricion_raiz`

### 5.3 Reglas de cuidados (enfermedad → acciones)
- `oidio` → mejorar ventilación, reducir humedad, fungicida suave
- `mancha_foliar` → retirar hojas afectadas, fungicida suave
- `pudricion_raiz` → ajustar riego, evitar encharcamiento

In [6]:
rules: List[Rule] = []

# --- Identificación (prioridad media) ---
rules += [
    Rule(
        nombre="Detecta_Cactus",
        antecedentes=[("$x","tiene_rasgo","tiene_espinas"),
                      ("$x","tiene_rasgo","tallo_suculento")],
        consecuentes=[("$x","es_planta","cactus")],
        prioridad=5
    ),
    Rule(
        nombre="Detecta_Suculenta",
        antecedentes=[("$x","tiene_rasgo","hojas_gruesas"),
                      ("$x","tiene_rasgo","roseta")],
        consecuentes=[("$x","es_planta","suculenta")],
        prioridad=5
    ),
    Rule(
        nombre="Detecta_Ficus",
        antecedentes=[("$x","tiene_rasgo","tallo_lenoso"),
                      ("$x","tiene_rasgo","hojas_ovales")],
        consecuentes=[("$x","es_planta","ficus")],
        prioridad=5
    ),
]

# --- Diagnóstico (prioridad alta) ---
rules += [
    Rule(
        nombre="Dx_Oidio",
        antecedentes=[("$x","tiene_sintoma","moho_blanco"),
                      ("$x","tiene_sintoma","hojas_deformadas")],
        consecuentes=[("$x","tiene_enfermedad","oidio")],
        prioridad=10
    ),
    Rule(
        nombre="Dx_Mancha_Foliar",
        antecedentes=[("$x","tiene_sintoma","manchas_circulares"),
                      ("$x","tiene_sintoma","puntos_negros")],
        consecuentes=[("$x","tiene_enfermedad","mancha_foliar")],
        prioridad=10
    ),
    Rule(
        nombre="Dx_Pudricion_Raiz",
        antecedentes=[("$x","tiene_sintoma","hojas_caidas"),
                      ("$x","tiene_sintoma","amarilleo"),
                      ("$x","tiene_contexto","exceso_humedad")],
        consecuentes=[("$x","tiene_enfermedad","pudricion_raiz")],
        prioridad=10
    ),
]

# --- Cuidados (prioridad baja, posterior al dx) ---
rules += [
    Rule(
        nombre="Care_Oidio",
        antecedentes=[("$x","tiene_enfermedad","oidio")],
        consecuentes=[("$x","requiere_cuidado","mejorar_ventilacion"),
                      ("$x","requiere_cuidado","reducir_humedad"),
                      ("$x","requiere_cuidado","fungicida_suave")],
        prioridad=1
    ),
    Rule(
        nombre="Care_Mancha_Foliar",
        antecedentes=[("$x","tiene_enfermedad","mancha_foliar")],
        consecuentes=[("$x","requiere_cuidado","retirar_hojas_afectadas"),
                      ("$x","requiere_cuidado","fungicida_suave")],
        prioridad=1
    ),
    Rule(
        nombre="Care_Pudricion_Raiz",
        antecedentes=[("$x","tiene_enfermedad","pudricion_raiz")],
        consecuentes=[("$x","requiere_cuidado","ajustar_riego"),
                      ("$x","requiere_cuidado","evitar_encharcamiento")],
        prioridad=1
    ),
]

len(rules), [r.nombre for r in rules]

(9,
 ['Detecta_Cactus',
  'Detecta_Suculenta',
  'Detecta_Ficus',
  'Dx_Oidio',
  'Dx_Mancha_Foliar',
  'Dx_Pudricion_Raiz',
  'Care_Oidio',
  'Care_Mancha_Foliar',
  'Care_Pudricion_Raiz'])

---

## 6) Ejemplo guiado (práctica): ejecutar el sistema experto

### Caso 1
Planta `p1`:
- rasgos: hojas_gruesas, roseta
- síntomas: moho_blanco, hojas_deformadas

Esperamos:
- tipo: suculenta
- enfermedad: oidio
- cuidados: mejorar_ventilación, reducir_humedad, fungicida_suave

In [7]:
def facts_of(sujeto: str, predicado: str, facts: Set[Fact]) -> List[str]:
    return sorted({f.objeto for f in facts if f.sujeto == sujeto and f.predicado == predicado})

facts0: Set[Fact] = {
    fact("p1","tiene_rasgo","hojas_gruesas"),
    fact("p1","tiene_rasgo","roseta"),
    fact("p1","tiene_sintoma","moho_blanco"),
    fact("p1","tiene_sintoma","hojas_deformadas"),
}

facts_all, trace = forward_chaining(rules, facts0, verbose=True)

print("\n--- RESULTADOS ---")
print("Tipo:", facts_of("p1","es_planta", facts_all))
print("Enfermedad:", facts_of("p1","tiene_enfermedad", facts_all))
print("Cuidados:", facts_of("p1","requiere_cuidado", facts_all))
print("\nTrazabilidad (primeros 5 pasos):")
for t in trace[:5]:
    print(t)

[Iter 1] FIRE: Dx_Oidio (prio=10) -> Fact(sujeto='p1', predicado='tiene_enfermedad', objeto='oidio') con {'$x': 'p1'}
[Iter 2] FIRE: Detecta_Suculenta (prio=5) -> Fact(sujeto='p1', predicado='es_planta', objeto='suculenta') con {'$x': 'p1'}
[Iter 3] FIRE: Care_Oidio (prio=1) -> Fact(sujeto='p1', predicado='requiere_cuidado', objeto='mejorar_ventilacion') con {'$x': 'p1'}
[Iter 4] FIRE: Care_Oidio (prio=1) -> Fact(sujeto='p1', predicado='requiere_cuidado', objeto='reducir_humedad') con {'$x': 'p1'}
[Iter 5] FIRE: Care_Oidio (prio=1) -> Fact(sujeto='p1', predicado='requiere_cuidado', objeto='fungicida_suave') con {'$x': 'p1'}
Detenido: no hay reglas aplicables (iter 6).

--- RESULTADOS ---
Tipo: ['suculenta']
Enfermedad: ['oidio']
Cuidados: ['fungicida_suave', 'mejorar_ventilacion', 'reducir_humedad']

Trazabilidad (primeros 5 pasos):
{'iter': 1, 'regla': 'Dx_Oidio', 'prioridad': 10, 'bindings': {'$x': 'p1'}, 'nuevo_hecho': Fact(sujeto='p1', predicado='tiene_enfermedad', objeto='oidio')}

### ¿Qué se observa en la trazabilidad?

Cada paso incluye:
- iteración
- regla disparada
- prioridad (select)
- bindings (quién es $x)
- nuevo hecho agregado

Esto implementa un **módulo de explicación** básico.

---

## 7) Reglas heurísticas y condicionales (control)

Ahora añadiremos un ejemplo:

### Heurística:
- Si hay `exceso_humedad`, sugiere “revisar drenaje” (aunque no sea un diagnóstico definitivo).

### Condicional / control:
- Si ya se diagnosticó `pudricion_raiz`, entonces **evitar** recomendar “aumentar riego”.
  (lo hacemos como regla de control que agrega un hecho `bloquea_cuidado(aumentar_riego)`).

Este tipo de reglas no son “del dominio” puro, sino de **gestión del razonamiento**.

In [8]:
# Heurística: contexto -> recomendación preventiva
rules.append(Rule(
    nombre="Heuristica_ExcesoHumedad",
    antecedentes=[("$x","tiene_contexto","exceso_humedad")],
    consecuentes=[("$x","requiere_cuidado","revisar_drenaje")],
    prioridad=2
))

# Control: si pudrición de raíz, bloquear "aumentar_riego"
rules.append(Rule(
    nombre="Control_Bloquea_AumentarRiego",
    antecedentes=[("$x","tiene_enfermedad","pudricion_raiz")],
    consecuentes=[("$x","bloquea_cuidado","aumentar_riego")],
    prioridad=3
))

# Regla que normalmente recomendaría aumentar riego si hay hojas_caidas (solo para demostrar el bloqueo)
rules.append(Rule(
    nombre="Heuristica_HojasCaidas_AumentarRiego",
    antecedentes=[("$x","tiene_sintoma","hojas_caidas")],
    consecuentes=[("$x","requiere_cuidado","aumentar_riego")],
    prioridad=2
))

# Regla condicional extra: si está bloqueado, NO agregar aumentar_riego
def not_blocked(bindings: Dict[str,str], facts: Set[Fact]) -> bool:
    x = bindings["$x"]
    return fact(x,"bloquea_cuidado","aumentar_riego") not in facts

# Reemplazamos la regla anterior por una versión condicional (más realista)
# (en clase puedes discutir por qué conviene separar "hechos de bloqueo" vs "eliminar hechos")
rules[-1] = Rule(
    nombre="Heuristica_HojasCaidas_AumentarRiego_condicional",
    antecedentes=[("$x","tiene_sintoma","hojas_caidas")],
    consecuentes=[("$x","requiere_cuidado","aumentar_riego")],
    prioridad=2,
    condicion_extra=not_blocked
)

len(rules)

12

### Caso 2 (para ver heurística + control)

Planta `p2`:
- síntomas: hojas_caidas, amarilleo
- contexto: exceso_humedad

Esperamos:
- diagnóstico: pudricion_raiz
- cuidados: ajustar_riego, evitar_encharcamiento, revisar_drenaje
- **NO** recomendar “aumentar_riego” (bloqueado)

In [9]:
facts_case2: Set[Fact] = {
    fact("p2","tiene_sintoma","hojas_caidas"),
    fact("p2","tiene_sintoma","amarilleo"),
    fact("p2","tiene_contexto","exceso_humedad"),
}

facts2, trace2 = forward_chaining(rules, facts_case2, verbose=False)

print("Enfermedad p2:", facts_of("p2","tiene_enfermedad", facts2))
print("Cuidados p2:", facts_of("p2","requiere_cuidado", facts2))
print("Bloqueos p2:", facts_of("p2","bloquea_cuidado", facts2))
print("¿Recomendó aumentar_riego?:", "aumentar_riego" in facts_of("p2","requiere_cuidado", facts2))

Enfermedad p2: ['pudricion_raiz']
Cuidados p2: ['ajustar_riego', 'evitar_encharcamiento', 'revisar_drenaje']
Bloqueos p2: ['aumentar_riego']
¿Recomendó aumentar_riego?: False


---

## 8) Ejercicios (para evaluación práctica)

> Estos ejercicios son exactamente la “aplicación de la IA simbólica” en un sistema experto.

### Ejercicio 1 (reglas de producción)
Agrega una nueva enfermedad `deficiencia_nutriente`:
- antecedentes: `amarilleo` AND `crecimiento_lento`
- consecuente: `tiene_enfermedad(deficiencia_nutriente)`

Luego agrega cuidados:
- `abonar_equilibrado`, `revisar_ph_suelo`

### Ejercicio 2 (heurísticas)
Crea una heurística:
- Si `poca_luz` entonces recomendar `mover_a_luz_indirecta`.

### Ejercicio 3 (control / condicional)
Crea una regla de control:
- Si `oidio`, bloquear `aumentar_humedad`.

Y modifica (o crea) una recomendación que solo se aplique si NO está bloqueada.

### Ejercicio 4 (explicabilidad)
Escribe una función `explicar(trace, sujeto)` que imprima:
- reglas que dispararon para ese sujeto
- en qué orden
- qué hecho agregaron

### Ejercicio 5 (diseño conceptual)
Dibuja el diagrama de componentes:
- Base de conocimiento
- Memoria de trabajo
- Motor de inferencia
- Interfaz (menciona voz como opción)
- Explicación

y describe el flujo de datos.

---

## 9) Reto integrador (mini-proyecto de 1 semana)

Construye una “interfaz” simple (texto) donde el usuario:
1) elige rasgos y síntomas de una lista (menú),
2) el sistema corre inferencia,
3) imprime diagnóstico y cuidados,
4) imprime explicación (trace).

**Extra (si ya viste Unidad II)**:
- Agrega TTS (pyttsx3) para leer el resultado.

In [10]:
# Plantilla de interfaz (texto) para el reto integrador

def diagnosticar_simple(sujeto: str, rasgos: List[str], sintomas: List[str], contexto: List[str]) -> Dict[str, Any]:
    f: Set[Fact] = set()
    for r in rasgos:
        f.add(fact(sujeto,"tiene_rasgo",r))
    for s in sintomas:
        f.add(fact(sujeto,"tiene_sintoma",s))
    for c in contexto:
        f.add(fact(sujeto,"tiene_contexto",c))
    allf, trace = forward_chaining(rules, f, verbose=False)
    return {
        "tipo": facts_of(sujeto,"es_planta",allf),
        "enfermedad": facts_of(sujeto,"tiene_enfermedad",allf),
        "cuidados": facts_of(sujeto,"requiere_cuidado",allf),
        "trace": trace
    }

def explicar(trace: List[Dict[str, Any]], sujeto: str, max_lines: int = 50):
    print(f"=== Explicación para {sujeto} ===")
    n = 0
    for step in trace:
        x = step["bindings"].get("$x")
        if x == sujeto:
            print(f"- Iter {step['iter']}: {step['regla']} -> {step['nuevo_hecho']}")
            n += 1
            if n >= max_lines:
                print("... (recortado)")
                break

# Demo del reto integrador:
resultado = diagnosticar_simple(
    sujeto="p_demo",
    rasgos=["hojas_gruesas","roseta"],
    sintomas=["moho_blanco","hojas_deformadas"],
    contexto=["exceso_humedad"]
)
print("Tipo:", resultado["tipo"])
print("Enfermedad:", resultado["enfermedad"])
print("Cuidados:", resultado["cuidados"])
explicar(resultado["trace"], "p_demo")

Tipo: ['suculenta']
Enfermedad: ['oidio']
Cuidados: ['fungicida_suave', 'mejorar_ventilacion', 'reducir_humedad', 'revisar_drenaje']
=== Explicación para p_demo ===
- Iter 1: Dx_Oidio -> Fact(sujeto='p_demo', predicado='tiene_enfermedad', objeto='oidio')
- Iter 2: Detecta_Suculenta -> Fact(sujeto='p_demo', predicado='es_planta', objeto='suculenta')
- Iter 3: Heuristica_ExcesoHumedad -> Fact(sujeto='p_demo', predicado='requiere_cuidado', objeto='revisar_drenaje')
- Iter 4: Care_Oidio -> Fact(sujeto='p_demo', predicado='requiere_cuidado', objeto='mejorar_ventilacion')
- Iter 5: Care_Oidio -> Fact(sujeto='p_demo', predicado='requiere_cuidado', objeto='reducir_humedad')
- Iter 6: Care_Oidio -> Fact(sujeto='p_demo', predicado='requiere_cuidado', objeto='fungicida_suave')


---

## 10) Cierre: conexión con Unidades I–III

- **Unidad I**: el SE puede ser el “módulo deliberativo” de un agente híbrido.  
- **Unidad II**: la interfaz puede ser por **voz** (STT/TTS), pero el razonamiento sigue siendo simbólico.  
- **Unidad III**: reglas + lógica + encadenamiento + ontologías (conocimiento estructurado).  
- **Unidad IV**: integra todo en un **Sistema Experto** funcional, explicable y evaluable.

el SE se puede:
- conectar con visión artificial para extraer rasgos,
- o integrar un LLM como interfaz natural,
- manteniendo el motor lógico como verificador/explicador.